<a href="https://colab.research.google.com/github/shubzz-dev/Smart-Loan-prediction-AI---logistic-regression/blob/main/capstone_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Smart Loan Assessment** is a machine-learning-based system that evaluates a customer's financial and credit profile to estimate their loan approval probability, creditworthiness score, risk level, and recommendation.

How the Model Works

The system uses Logistic Regression trained on the German Credit Dataset. Customer information such as loan duration, credit amount, credit history, savings, employment, housing, existing credits, and other financial factors is collected. Numerical features are scaled, while categorical features are converted using One-Hot Encoding. The trained model then calculates the probability that the applicant belongs to the Good Credit class.

The probability is converted into a 0–100 creditworthiness score:

Probability × 100 = Creditworthiness Score
Based on this score and the configured 0.50 approval threshold, the system determines whether the loan should be approved, rejected, or given further consideration.

# CELL 1 — Install packages

In [1]:
# ============================================================
# CELL 1: INSTALL REQUIRED PACKAGES
# ============================================================

!pip install -q streamlit pandas numpy scikit-learn matplotlib seaborn requests

# Download Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb

# IMPORTANT:
# Do NOT use -q with dpkg
!dpkg -i cloudflared-linux-amd64.deb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 44.7 MB/s eta 0:00:00
Selecting previously unselected package cloudflared.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.7.3) ...
Setting up cloudflared (2026.7.3) ...
Processing triggers for man-db (2.10.2-1) ...


# Import libraries and create folders

In [2]:
# ============================================================
# CELL 2: IMPORT LIBRARIES + CREATE DIRECTORIES
# ============================================================

import os
import zipfile
import pickle
import requests

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Create folders
os.makedirs("/content/dataset", exist_ok=True)
os.makedirs("/content/model_artifacts", exist_ok=True)

print("Libraries imported successfully.")
print("Directories created successfully.")

Libraries imported successfully.
Directories created successfully.


# Download German Credit Dataset

In [3]:
# ============================================================
# CELL 3: DOWNLOAD GERMAN CREDIT DATASET
# ============================================================

dataset_url = (
    "https://archive.ics.uci.edu/static/public/144/"
    "statlog+german+credit+data.zip"
)

zip_path = "/content/dataset/german_credit.zip"

print("=" * 70)
print("DOWNLOADING GERMAN CREDIT DATASET")
print("=" * 70)

response = requests.get(dataset_url)

if response.status_code == 200:

    with open(zip_path, "wb") as f:
        f.write(response.content)

    print("✅ Dataset downloaded successfully.")

else:

    print("❌ Dataset download failed.")
    print("HTTP Status:", response.status_code)

DOWNLOADING GERMAN CREDIT DATASET
✅ Dataset downloaded successfully.


# Extract Dataset

In [4]:
# ============================================================
# CELL 4: EXTRACT DATASET
# ============================================================

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    zip_ref.extractall("/content/dataset")


print("=" * 70)
print("EXTRACTED FILES")
print("=" * 70)

for root, dirs, files in os.walk("/content/dataset"):

    for file in files:

        print(os.path.join(root, file))

EXTRACTED FILES
/content/dataset/german.data
/content/dataset/Index
/content/dataset/german.data-numeric
/content/dataset/german_credit.zip
/content/dataset/german.doc


# Load Dataset

In [5]:
# ============================================================
# CELL 5: LOAD GERMAN CREDIT DATASET
# ============================================================

german_data_path = "/content/dataset/german.data"

columns = [

    "Status_of_existing_checking_account",

    "Duration_in_month",

    "Credit_history",

    "Purpose",

    "Credit_amount",

    "Savings_account_bonds",

    "Present_employment_since",

    "Installment_rate_in_percentage_of_disposable_income",

    "Personal_status_and_sex",

    "Other_debtors_guarantors",

    "Present_residence_since",

    "Property",

    "Age_in_years",

    "Other_installment_plans",

    "Housing",

    "Number_of_existing_credits_at_this_bank",

    "Job",

    "Number_of_people_being_liable_to_provide_maintenance_for",

    "Telephone",

    "Foreign_worker",

    "Target"
]


df = pd.read_csv(

    german_data_path,

    sep=r"\s+",

    header=None,

    names=columns

)


print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("Shape:", df.shape)

display(df.head())

DATASET INFORMATION
Shape: (1000, 21)


,Status_of_existing_checking_account,Duration_in_month,Credit_history,Purpose,Credit_amount,Savings_account_bonds,Present_employment_since,Installment_rate_in_percentage_of_disposable_income,Personal_status_and_sex,Other_debtors_guarantors,...,Property,Age_in_years,Other_installment_plans,Housing,Number_of_existing_credits_at_this_bank,Job,Number_of_people_being_liable_to_provide_maintenance_for,Telephone,Foreign_worker,Target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


# Prepare Target

In [6]:
# ============================================================
# CELL 6: PREPARE TARGET VARIABLE
# ============================================================

# Original German Credit Dataset:
#
# 1 = Good Credit
# 2 = Bad Credit
#
# We convert:
#
# Good Credit = 1
# Bad Credit  = 0

df["Target"] = df["Target"].map({

    1: 1,
    2: 0

})


print("=" * 70)
print("TARGET DISTRIBUTION")
print("=" * 70)

print(df["Target"].value_counts())

print("\nPercentage distribution:")

print(

    df["Target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)

)

TARGET DISTRIBUTION
Target
1    700
0    300
Name: count, dtype: int64

Percentage distribution:
Target
1    70.0
0    30.0
Name: proportion, dtype: float64


# Define Numerical and Categorical Columns

In [7]:
# ============================================================
# CELL 7: DEFINE FEATURES
# ============================================================

numerical_cols = [

    "Duration_in_month",

    "Credit_amount",

    "Installment_rate_in_percentage_of_disposable_income",

    "Present_residence_since",

    "Age_in_years",

    "Number_of_existing_credits_at_this_bank",

    "Number_of_people_being_liable_to_provide_maintenance_for"

]


categorical_cols = [

    "Status_of_existing_checking_account",

    "Credit_history",

    "Purpose",

    "Savings_account_bonds",

    "Present_employment_since",

    "Personal_status_and_sex",

    "Other_debtors_guarantors",

    "Property",

    "Other_installment_plans",

    "Housing",

    "Job",

    "Telephone",

    "Foreign_worker"

]


print("Numerical columns:")
print(numerical_cols)

print("\nCategorical columns:")
print(categorical_cols)

print("\nTotal original features:")

print(

    len(numerical_cols) +
    len(categorical_cols)

)

Numerical columns:
['Duration_in_month', 'Credit_amount', 'Installment_rate_in_percentage_of_disposable_income', 'Present_residence_since', 'Age_in_years', 'Number_of_existing_credits_at_this_bank', 'Number_of_people_being_liable_to_provide_maintenance_for']

Categorical columns:
['Status_of_existing_checking_account', 'Credit_history', 'Purpose', 'Savings_account_bonds', 'Present_employment_since', 'Personal_status_and_sex', 'Other_debtors_guarantors', 'Property', 'Other_installment_plans', 'Housing', 'Job', 'Telephone', 'Foreign_worker']

Total original features:
20


# Create X and y + Train/Test Split

In [8]:
# ============================================================
# CELL 8: TRAIN / TEST SPLIT
# ============================================================

X = df[
    numerical_cols + categorical_cols
].copy()

y = df["Target"].copy()


X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)


print("=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

print("Total samples:", len(X))

print("Training samples:", len(X_train))

print("Testing samples:", len(X_test))

print("\nTraining target distribution:")

print(y_train.value_counts())

print("\nTesting target distribution:")

print(y_test.value_counts())

TRAIN / TEST SPLIT
Total samples: 1000
Training samples: 800
Testing samples: 200

Training target distribution:
Target
1    560
0    240
Name: count, dtype: int64

Testing target distribution:
Target
1    140
0     60
Name: count, dtype: int64


# One-Hot Encoding

In [9]:
# ============================================================
# CELL 9: ONE-HOT ENCODING
# ============================================================

X_train_encoded = pd.get_dummies(

    X_train,

    columns=categorical_cols

)


X_test_encoded = pd.get_dummies(

    X_test,

    columns=categorical_cols

)


# Make sure test has exactly the same columns as training

encoded_feature_columns = X_train_encoded.columns.tolist()


X_test_encoded = X_test_encoded.reindex(

    columns=encoded_feature_columns,

    fill_value=0

)


print("=" * 70)
print("ENCODING COMPLETE")
print("=" * 70)

print("Training shape:", X_train_encoded.shape)

print("Testing shape:", X_test_encoded.shape)

print("Encoded features:", len(encoded_feature_columns))

ENCODING COMPLETE
Training shape: (800, 61)
Testing shape: (200, 61)
Encoded features: 61


# Scale Numerical Features

In [10]:
# ============================================================
# CELL 10: SCALE NUMERICAL FEATURES
# ============================================================

scaler = StandardScaler()


X_train_scaled = X_train_encoded.copy()

X_test_scaled = X_test_encoded.copy()


# Scale only numerical columns
X_train_scaled[numerical_cols] = scaler.fit_transform(

    X_train_encoded[numerical_cols]

)


X_test_scaled[numerical_cols] = scaler.transform(

    X_test_encoded[numerical_cols]

)


print("=" * 70)
print("FEATURE SCALING COMPLETE")
print("=" * 70)

print("Numerical features scaled successfully.")

FEATURE SCALING COMPLETE
Numerical features scaled successfully.


# Train Logistic Regression

In [11]:
# ============================================================
# CELL 11: TRAIN LOGISTIC REGRESSION
# ============================================================

print("=" * 70)
print("TRAINING LOGISTIC REGRESSION")
print("=" * 70)


model = LogisticRegression(

    max_iter=5000,

    solver="lbfgs",

    random_state=42

)


model.fit(

    X_train_scaled,

    y_train

)


print("✅ Model training completed successfully.")

TRAINING LOGISTIC REGRESSION
✅ Model training completed successfully.


# Test Model

In [12]:
# ============================================================
# CELL 12: MODEL TESTING
# ============================================================

y_pred = model.predict(

    X_test_scaled

)


y_probability = model.predict_proba(

    X_test_scaled

)[:, 1]


accuracy = accuracy_score(

    y_test,

    y_pred

)


roc_auc = roc_auc_score(

    y_test,

    y_probability

)


print("=" * 70)
print("MODEL TEST RESULTS")
print("=" * 70)

print(

    f"Test Accuracy: {accuracy * 100:.2f}%"

)

print(

    f"ROC-AUC Score: {roc_auc:.4f}"

)

MODEL TEST RESULTS
Test Accuracy: 70.50%
ROC-AUC Score: 0.7594


In [13]:
# ============================================================
# CELL 13: CLASSIFICATION REPORT
# ============================================================

print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)


print(

    classification_report(

        y_test,

        y_pred,

        target_names=[

            "Bad Credit",

            "Good Credit"

        ]

    )

)


print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)


cm = confusion_matrix(

    y_test,

    y_pred

)


print(cm)

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Bad Credit       0.51      0.45      0.48        60
 Good Credit       0.78      0.81      0.79       140

    accuracy                           0.70       200
   macro avg       0.64      0.63      0.64       200
weighted avg       0.70      0.70      0.70       200

CONFUSION MATRIX
[[ 27  33]
 [ 26 114]]


In [14]:
# ============================================================
# CELL 14: APPROVAL THRESHOLD
# ============================================================

# Probability >= 0.50
#       -> Loan Approved
#
# Probability < 0.50
#       -> Loan Not Approved

approval_threshold = 0.50


print("=" * 70)
print("APPROVAL RULE")
print("=" * 70)

print(

    "Approval threshold:",

    approval_threshold

)

print(

    "Approval score:",

    approval_threshold * 100,

    "/ 100"

)

APPROVAL RULE
Approval threshold: 0.5
Approval score: 50.0 / 100


# Save All Model Artifacts

In [15]:
# ============================================================
# CELL 15: SAVE MODEL ARTIFACTS
# ============================================================

artifact_path = "/content/model_artifacts"


# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

with open(

    f"{artifact_path}/logistic_regression_model.pkl",

    "wb"

) as f:

    pickle.dump(model, f)


# ------------------------------------------------------------
# SCALER
# ------------------------------------------------------------

with open(

    f"{artifact_path}/scaler.pkl",

    "wb"

) as f:

    pickle.dump(scaler, f)


# ------------------------------------------------------------
# CATEGORICAL COLUMNS
# ------------------------------------------------------------

with open(

    f"{artifact_path}/categorical_cols.pkl",

    "wb"

) as f:

    pickle.dump(categorical_cols, f)


# ------------------------------------------------------------
# NUMERICAL COLUMNS
# ------------------------------------------------------------

with open(

    f"{artifact_path}/numerical_cols.pkl",

    "wb"

) as f:

    pickle.dump(numerical_cols, f)


# ------------------------------------------------------------
# ENCODED FEATURES
# ------------------------------------------------------------

with open(

    f"{artifact_path}/encoded_feature_columns.pkl",

    "wb"

) as f:

    pickle.dump(encoded_feature_columns, f)


# ------------------------------------------------------------
# APPROVAL THRESHOLD
# ------------------------------------------------------------

with open(

    f"{artifact_path}/approval_threshold.pkl",

    "wb"

) as f:

    pickle.dump(approval_threshold, f)


# ------------------------------------------------------------
# MODEL INFORMATION
# ------------------------------------------------------------

model_info = {

    "model_name": "Logistic Regression",

    "dataset": "German Credit Dataset",

    "training_samples": len(X_train),

    "testing_samples": len(X_test),

    "accuracy": float(accuracy),

    "roc_auc": float(roc_auc),

    "approval_threshold": float(approval_threshold),

    "number_of_features": len(encoded_feature_columns)

}


with open(

    f"{artifact_path}/model_info.pkl",

    "wb"

) as f:

    pickle.dump(model_info, f)


print("=" * 70)
print("MODEL ARTIFACTS SAVED")
print("=" * 70)


for file in os.listdir(artifact_path):

    print("✅", file)

MODEL ARTIFACTS SAVED
✅ logistic_regression_model.pkl
✅ approval_threshold.pkl
✅ encoded_feature_columns.pkl
✅ scaler.pkl
✅ numerical_cols.pkl
✅ model_info.pkl
✅ categorical_cols.pkl


In [16]:
# ============================================================
# CELL 16: VERIFY MODEL ARTIFACTS
# ============================================================

print("=" * 70)
print("VERIFYING MODEL ARTIFACTS")
print("=" * 70)


required_files = [

    "logistic_regression_model.pkl",

    "scaler.pkl",

    "categorical_cols.pkl",

    "numerical_cols.pkl",

    "encoded_feature_columns.pkl",

    "approval_threshold.pkl",

    "model_info.pkl"

]


for file in required_files:

    path = f"/content/model_artifacts/{file}"

    if os.path.exists(path):

        print("✅", file)

    else:

        print("❌ MISSING:", file)

VERIFYING MODEL ARTIFACTS
✅ logistic_regression_model.pkl
✅ scaler.pkl
✅ categorical_cols.pkl
✅ numerical_cols.pkl
✅ encoded_feature_columns.pkl
✅ approval_threshold.pkl
✅ model_info.pkl


# Create Streamlit App

In [26]:
# ============================================================
# CELL 17 — CREATE NEON STREAMLIT APPLICATION
# ============================================================

%%writefile /content/app.py

import streamlit as st
import pandas as pd
import pickle


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Smart Loan AI",
    page_icon="🏦",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ============================================================
# LOAD ARTIFACTS
# ============================================================

@st.cache_resource
def load_artifacts():

    base = "/content/model_artifacts"

    with open(
        f"{base}/logistic_regression_model.pkl",
        "rb"
    ) as f:
        model = pickle.load(f)

    with open(
        f"{base}/scaler.pkl",
        "rb"
    ) as f:
        scaler = pickle.load(f)

    with open(
        f"{base}/numerical_cols.pkl",
        "rb"
    ) as f:
        numerical_cols = pickle.load(f)

    with open(
        f"{base}/categorical_cols.pkl",
        "rb"
    ) as f:
        categorical_cols = pickle.load(f)

    with open(
        f"{base}/encoded_feature_columns.pkl",
        "rb"
    ) as f:
        encoded_feature_columns = pickle.load(f)

    with open(
        f"{base}/approval_threshold.pkl",
        "rb"
    ) as f:
        approval_threshold = pickle.load(f)

    with open(
        f"{base}/model_info.pkl",
        "rb"
    ) as f:
        model_info = pickle.load(f)

    return (
        model,
        scaler,
        numerical_cols,
        categorical_cols,
        encoded_feature_columns,
        approval_threshold,
        model_info
    )


(
    model,
    scaler,
    numerical_cols,
    categorical_cols,
    encoded_feature_columns,
    approval_threshold,
    model_info
) = load_artifacts()


# ============================================================
# NEON CSS
# ============================================================

st.markdown(
"""
<style>

/* =========================================================
   MAIN BACKGROUND
   ========================================================= */

.stApp {

    background:

        radial-gradient(
            circle at 10% 10%,
            rgba(0,255,255,0.12),
            transparent 28%
        ),

        radial-gradient(
            circle at 90% 10%,
            rgba(140,0,255,0.15),
            transparent 30%
        ),

        radial-gradient(
            circle at 50% 100%,
            rgba(0,100,255,0.12),
            transparent 35%
        ),

        #030712;

    color: white;

}


/* =========================================================
   GRID
   ========================================================= */

.stApp::before {

    content: "";

    position: fixed;

    inset: 0;

    background-image:

        linear-gradient(
            rgba(0,255,255,0.035) 1px,
            transparent 1px
        ),

        linear-gradient(
            90deg,
            rgba(0,255,255,0.035) 1px,
            transparent 1px
        );

    background-size: 45px 45px;

    animation: gridMove 15s linear infinite;

    pointer-events: none;

}


@keyframes gridMove {

    from {
        transform: translateY(0);
    }

    to {
        transform: translateY(45px);
    }

}


/* =========================================================
   MAIN CONTAINER
   ========================================================= */

.block-container {

    position: relative;

    z-index: 2;

    padding-top: 35px;

    max-width: 1450px;

}


/* =========================================================
   TITLE
   ========================================================= */

.neon-title {

    text-align: center;

    font-size: 52px;

    font-weight: 900;

    color: white;

    letter-spacing: 2px;

    text-shadow:

        0 0 5px #00ffff,

        0 0 15px #00ffff,

        0 0 30px #00ffff,

        0 0 60px #7a00ff;

    animation: titlePulse 2s infinite alternate;

}


@keyframes titlePulse {

    from {

        text-shadow:

            0 0 5px #00ffff,

            0 0 15px #00ffff;

    }

    to {

        text-shadow:

            0 0 10px #00ffff,

            0 0 25px #00ffff,

            0 0 50px #7a00ff;

    }

}


.subtitle {

    text-align: center;

    color: #9caac8;

    font-size: 18px;

    margin-bottom: 30px;

}


.badge {

    text-align: center;

    color: #00ffff;

    font-weight: 800;

    letter-spacing: 3px;

    font-size: 13px;

    margin-bottom: 12px;

}


/* =========================================================
   SECTION HEADINGS
   ========================================================= */

.section-heading {

    color: white;

    font-size: 25px;

    font-weight: 800;

    border-left: 4px solid #00ffff;

    padding-left: 12px;

    margin-top: 30px;

    margin-bottom: 20px;

    text-shadow:

        0 0 10px rgba(0,255,255,0.7);

}


/* =========================================================
   INPUTS
   ========================================================= */

label {

    color: #dce6ff !important;

    font-weight: 600 !important;

}


div[data-baseweb="input"] {

    background: #080d20 !important;

    border: 1px solid rgba(0,255,255,0.35) !important;

    border-radius: 10px !important;

}


div[data-baseweb="input"]:focus-within {

    border-color: #00ffff !important;

    box-shadow:

        0 0 10px rgba(0,255,255,0.5);

}


input {

    color: white !important;

}


/* =========================================================
   SELECTBOX
   ========================================================= */

div[data-baseweb="select"] > div {

    background: #080d20 !important;

    border: 1px solid rgba(0,255,255,0.35) !important;

    border-radius: 10px !important;

}


div[data-baseweb="select"] span {

    color: white !important;

}


/* =========================================================
   BUTTON
   ========================================================= */

.stButton > button {

    background:

        linear-gradient(
            90deg,
            #00c6ff,
            #7a00ff
        );

    color: white !important;

    border: 1px solid #00ffff;

    border-radius: 12px;

    font-size: 18px;

    font-weight: 900;

    padding: 15px;

    box-shadow:

        0 0 12px rgba(0,255,255,0.5),

        0 0 30px rgba(122,0,255,0.3);

    transition: all 0.3s ease;

}


.stButton > button:hover {

    transform: translateY(-3px);

    box-shadow:

        0 0 20px #00ffff,

        0 0 40px #7a00ff;

}


/* =========================================================
   METRIC CARDS
   ========================================================= */

div[data-testid="stMetric"] {

    background:

        linear-gradient(
            145deg,
            rgba(8,15,35,0.95),
            rgba(10,8,35,0.95)
        );

    border: 1px solid rgba(0,255,255,0.35);

    border-radius: 18px;

    padding: 20px;

    box-shadow:

        0 0 15px rgba(0,255,255,0.12);

    transition: 0.3s ease;

}


div[data-testid="stMetric"]:hover {

    transform: translateY(-5px);

    border-color: #00ffff;

    box-shadow:

        0 0 25px rgba(0,255,255,0.4);

}


div[data-testid="stMetricLabel"] {

    color: #8e9bb8 !important;

}


div[data-testid="stMetricValue"] {

    color: #00ffff !important;

    font-weight: 900 !important;

    text-shadow:

        0 0 10px #00ffff;

}


/* =========================================================
   SCORE BOX
   ========================================================= */

.score-box {

    text-align: center;

    padding: 30px;

    margin: 25px 0;

    background:

        linear-gradient(
            135deg,
            rgba(0,255,255,0.07),
            rgba(122,0,255,0.10)
        );

    border: 1px solid #00ffff;

    border-radius: 22px;

    box-shadow:

        0 0 20px rgba(0,255,255,0.25);

}


.score-label {

    color: #9caac8;

    font-size: 15px;

    letter-spacing: 3px;

    font-weight: 700;

}


.score-value {

    font-size: 75px;

    font-weight: 900;

    color: #00ffff;

    text-shadow:

        0 0 10px #00ffff,

        0 0 25px #00ffff,

        0 0 50px #0088ff;

    animation: scorePulse 2s infinite;

}


@keyframes scorePulse {

    0%, 100% {

        transform: scale(1);

    }

    50% {

        transform: scale(1.04);

    }

}


/* =========================================================
   SIDEBAR
   ========================================================= */

section[data-testid="stSidebar"] {

    background:

        linear-gradient(
            180deg,
            #050817,
            #08031a
        );

    border-right:

        1px solid rgba(0,255,255,0.25);

}


section[data-testid="stSidebar"] * {

    color: #dce6ff !important;

}


/* =========================================================
   PROGRESS
   ========================================================= */

div[data-testid="stProgress"] > div > div {

    background:

        linear-gradient(
            90deg,
            #00ffff,
            #0088ff,
            #9b00ff
        ) !important;

    box-shadow:

        0 0 15px #00ffff;

}


/* =========================================================
   FOOTER
   ========================================================= */

.footer {

    text-align: center;

    color: #64748b;

    padding: 30px;

}


/* =========================================================
   HIDE DEFAULT STREAMLIT
   ========================================================= */

#MainMenu {

    visibility: hidden;

}

footer {

    visibility: hidden;

}

header {

    visibility: hidden;

}

</style>
""",
unsafe_allow_html=True
)


# ============================================================
# HEADER
# ============================================================

st.markdown(
    '<div class="badge">⚡ AI CREDIT INTELLIGENCE SYSTEM ⚡</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="neon-title">🏦 SMART LOAN AI</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'Machine Learning based Creditworthiness & Loan Assessment'
    '</div>',
    unsafe_allow_html=True
)


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.markdown("## 🧠 MODEL STATUS")

    st.success("● MODEL ONLINE")

    st.markdown("---")

    st.markdown("## 📊 CREDIT SCORE GUIDE")

    st.write("🟢 **80–100** — Excellent")

    st.write("🟢 **65–79** — Good")

    st.write("🟡 **50–64** — Moderate")

    st.write("🟠 **30–49** — Weak")

    st.write("🔴 **0–29** — Poor")

    st.markdown("---")

    st.markdown("## ⚙️ MODEL INFORMATION")

    st.write(
        "**Algorithm:** Logistic Regression"
    )

    st.write(
        "**Dataset:** German Credit Dataset"
    )

    st.write(
        f"**Accuracy:** "
        f"{model_info['accuracy'] * 100:.2f}%"
    )

    st.write(
        f"**ROC-AUC:** "
        f"{model_info['roc_auc']:.4f}"
    )

    st.write(
        f"**Features:** "
        f"{model_info['number_of_features']}"
    )

    st.write(
        f"**Approval Threshold:** "
        f"{approval_threshold * 100:.0f}%"
    )

    st.markdown("---")

    st.info(
        "This is an ML-derived creditworthiness "
        "score and is not an official CIBIL score."
    )


# ============================================================
# FINANCIAL INFORMATION
# ============================================================

st.markdown(
    '<div class="section-heading">'
    '💰 FINANCIAL & LOAN INFORMATION'
    '</div>',
    unsafe_allow_html=True
)


col1, col2, col3 = st.columns(3)


with col1:

    duration = st.number_input(
        "Loan Duration (months)",
        min_value=1,
        max_value=100,
        value=24,
        step=1
    )


with col2:

    credit_amount = st.number_input(
        "Credit Amount",
        min_value=0,
        max_value=100000,
        value=5000,
        step=100
    )


with col3:

    installment_rate = st.slider(
        "Installment Rate (% of Income)",
        min_value=1,
        max_value=4,
        value=2
    )


col1, col2, col3 = st.columns(3)


with col1:

    residence = st.number_input(
        "Residence Since (years)",
        min_value=1,
        max_value=10,
        value=2
    )


with col2:

    age = st.number_input(
        "Age (years)",
        min_value=18,
        max_value=100,
        value=30
    )


with col3:

    existing_credits = st.number_input(
        "Existing Credits at This Bank",
        min_value=0,
        max_value=10,
        value=1
    )


dependents = st.number_input(
    "People Dependent on Applicant",
    min_value=0,
    max_value=10,
    value=1
)


# ============================================================
# CUSTOMER PROFILE
# ============================================================

st.markdown(
    '<div class="section-heading">'
    '👤 CUSTOMER CREDIT PROFILE'
    '</div>',
    unsafe_allow_html=True
)


# ------------------------------------------------------------
# CHECKING ACCOUNT
# ------------------------------------------------------------

checking_options = {

    "No checking account": "A14",

    "Balance below 0": "A11",

    "Balance between 0 and 200": "A12",

    "Balance above 200": "A13"

}


checking_display = st.selectbox(
    "Existing Checking Account",
    list(checking_options.keys())
)

checking_account = checking_options[
    checking_display
]


# ------------------------------------------------------------
# CREDIT HISTORY
# ------------------------------------------------------------

credit_history_options = {

    "No credits / all paid duly": "A30",

    "All credits paid duly": "A31",

    "Existing credits paid duly": "A32",

    "Delay in payment in the past": "A33",

    "Critical / other existing credits": "A34"

}


credit_history_display = st.selectbox(
    "Credit History",
    list(credit_history_options.keys())
)

credit_history = credit_history_options[
    credit_history_display
]


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------

purpose_options = {

    "New car": "A40",

    "Used car": "A41",

    "Furniture / equipment": "A42",

    "Radio / television": "A43",

    "Domestic appliances": "A44",

    "Repairs": "A45",

    "Education": "A46",

    "Vacation": "A48",

    "Business": "A49",

    "Other": "A410"

}


purpose_display = st.selectbox(
    "Purpose of Loan",
    list(purpose_options.keys())
)

purpose = purpose_options[
    purpose_display
]


# ------------------------------------------------------------
# SAVINGS
# ------------------------------------------------------------

savings_options = {

    "Less than 100": "A61",

    "100 to 499": "A62",

    "500 to 999": "A63",

    "1000 or more": "A64",

    "Unknown / no savings": "A65"

}


savings_display = st.selectbox(
    "Savings Account / Bonds",
    list(savings_options.keys())
)

savings = savings_options[
    savings_display
]


# ------------------------------------------------------------
# EMPLOYMENT
# ------------------------------------------------------------

employment_options = {

    "Unemployed": "A71",

    "Less than 1 year": "A72",

    "1 to 4 years": "A73",

    "4 to 7 years": "A74",

    "7 years or more": "A75"

}


employment_display = st.selectbox(
    "Employment Duration",
    list(employment_options.keys())
)

employment = employment_options[
    employment_display
]


# ------------------------------------------------------------
# PERSONAL STATUS
# ------------------------------------------------------------

personal_options = {

    "Male: divorced / separated": "A91",

    "Female: divorced / separated / married": "A92",

    "Male: single": "A93",

    "Male: married / widowed": "A94"

}


personal_display = st.selectbox(
    "Personal Status & Sex",
    list(personal_options.keys())
)

personal_status = personal_options[
    personal_display
]


# ------------------------------------------------------------
# DEBTORS
# ------------------------------------------------------------

debtors_options = {

    "None": "A101",

    "Co-applicant": "A102",

    "Guarantor": "A103"

}


debtors_display = st.selectbox(
    "Other Debtors / Guarantors",
    list(debtors_options.keys())
)

other_debtors = debtors_options[
    debtors_display
]


# ------------------------------------------------------------
# PROPERTY
# ------------------------------------------------------------

property_options = {

    "Real estate": "A121",

    "Life insurance": "A122",

    "Car / other property": "A123",

    "No property": "A124"

}


property_display = st.selectbox(
    "Property / Collateral",
    list(property_options.keys())
)

property_type = property_options[
    property_display
]


# ------------------------------------------------------------
# INSTALLMENT PLANS
# ------------------------------------------------------------

installment_options = {

    "None": "A141",

    "Bank": "A142",

    "Stores": "A143"

}


installment_display = st.selectbox(
    "Other Installment Plans",
    list(installment_options.keys())
)

installment_plans = installment_options[
    installment_display
]


# ------------------------------------------------------------
# HOUSING
# ------------------------------------------------------------

housing_options = {

    "Rent": "A151",

    "Own": "A152",

    "For free": "A153"

}


housing_display = st.selectbox(
    "Housing",
    list(housing_options.keys())
)

housing = housing_options[
    housing_display
]


# ------------------------------------------------------------
# JOB
# ------------------------------------------------------------

job_options = {

    "Unskilled / non-resident": "A171",

    "Unskilled / resident": "A172",

    "Skilled employee / official": "A173",

    "Management / self-employed / highly qualified": "A174"

}


job_display = st.selectbox(
    "Job Type",
    list(job_options.keys())
)

job = job_options[
    job_display
]


# ------------------------------------------------------------
# TELEPHONE
# ------------------------------------------------------------

telephone_options = {

    "No": "A191",

    "Yes": "A192"

}


telephone_display = st.selectbox(
    "Telephone",
    list(telephone_options.keys())
)

telephone = telephone_options[
    telephone_display
]


# ------------------------------------------------------------
# FOREIGN WORKER
# ------------------------------------------------------------

foreign_options = {

    "Yes": "A201",

    "No": "A202"

}


foreign_display = st.selectbox(
    "Foreign Worker",
    list(foreign_options.keys())
)

foreign_worker = foreign_options[
    foreign_display
]


# ============================================================
# ASSESS BUTTON
# ============================================================

st.markdown("---")

check = st.button(
    "⚡ RUN AI CREDIT ASSESSMENT ⚡",
    use_container_width=True
)


# ============================================================
# PREDICTION
# ============================================================

if check:

    # --------------------------------------------------------
    # INPUT DATAFRAME
    # --------------------------------------------------------

    input_data = pd.DataFrame({

        "Status_of_existing_checking_account":
            [checking_account],

        "Duration_in_month":
            [duration],

        "Credit_history":
            [credit_history],

        "Purpose":
            [purpose],

        "Credit_amount":
            [credit_amount],

        "Savings_account_bonds":
            [savings],

        "Present_employment_since":
            [employment],

        "Installment_rate_in_percentage_of_disposable_income":
            [installment_rate],

        "Personal_status_and_sex":
            [personal_status],

        "Other_debtors_guarantors":
            [other_debtors],

        "Present_residence_since":
            [residence],

        "Property":
            [property_type],

        "Age_in_years":
            [age],

        "Other_installment_plans":
            [installment_plans],

        "Housing":
            [housing],

        "Number_of_existing_credits_at_this_bank":
            [existing_credits],

        "Job":
            [job],

        "Number_of_people_being_liable_to_provide_maintenance_for":
            [dependents],

        "Telephone":
            [telephone],

        "Foreign_worker":
            [foreign_worker]

    })


    # --------------------------------------------------------
    # ONE-HOT ENCODING
    # --------------------------------------------------------

    input_encoded = pd.get_dummies(
        input_data,
        columns=categorical_cols
    )


    # --------------------------------------------------------
    # EXACT TRAINING COLUMNS
    # --------------------------------------------------------

    input_encoded = input_encoded.reindex(
        columns=encoded_feature_columns,
        fill_value=0
    )


    # --------------------------------------------------------
    # SCALE NUMERICAL FEATURES
    # --------------------------------------------------------

    input_scaled = input_encoded.copy()

    input_scaled[numerical_cols] = scaler.transform(
        input_encoded[numerical_cols]
    )


    # --------------------------------------------------------
    # PREDICTION
    # --------------------------------------------------------

    probability = model.predict_proba(
        input_scaled
    )[0][1]

    probability = float(probability)


    # --------------------------------------------------------
    # SCORE
    # --------------------------------------------------------

    credit_score = round(
        probability * 100,
        1
    )


    # --------------------------------------------------------
    # RATING
    # --------------------------------------------------------

    if credit_score >= 80:

        rating = "Excellent"

        risk = "Low Risk"

        consideration = "Very High"

    elif credit_score >= 65:

        rating = "Good"

        risk = "Low–Moderate Risk"

        consideration = "High"

    elif credit_score >= 50:

        rating = "Moderate"

        risk = "Moderate Risk"

        consideration = "Further Review"

    elif credit_score >= 30:

        rating = "Weak"

        risk = "High Risk"

        consideration = "Low"

    else:

        rating = "Poor"

        risk = "Very High Risk"

        consideration = "Very Low"


    # ========================================================
    # RESULT
    # ========================================================

    st.markdown(
        '<div class="section-heading">'
        '📡 AI ASSESSMENT OUTPUT'
        '</div>',
        unsafe_allow_html=True
    )





    # ========================================================
    # RESULT CARDS
    # ========================================================

    c1, c2, c3, c4 = st.columns(4)


    with c1:

        st.metric(
            "GOOD CREDIT PROBABILITY",
            f"{probability * 100:.1f}%"
        )


    with c2:

        st.metric(
            "CREDIT RATING",
            rating
        )


    with c3:

        st.metric(
            "RISK LEVEL",
            risk
        )


    with c4:

        st.metric(
            "CONSIDERATION",
            consideration
        )


    # ========================================================
    # PROGRESS
    # ========================================================

    st.markdown(
        "### 📈 CREDITWORTHINESS LEVEL"
    )

    st.progress(
        probability
    )


    # ========================================================
    # CONSIDERATION EXPLANATION
    # ========================================================

    if credit_score >= 80:

        st.success(
            "🟢 VERY HIGH CONSIDERATION — "
            "The applicant has an excellent model-derived "
            "creditworthiness score."
        )

    elif credit_score >= 65:

        st.success(
            "🟢 HIGH CONSIDERATION — "
            "The applicant demonstrates a good "
            "creditworthiness profile."
        )

    elif credit_score >= 50:

        st.warning(
            "🟡 FURTHER REVIEW — "
            "The applicant has moderate creditworthiness. "
            "Additional verification may be appropriate."
        )

    elif credit_score >= 30:

        st.warning(
            "🟠 LOW CONSIDERATION — "
            "The applicant represents a comparatively "
            "higher-risk profile."
        )

    else:

        st.error(
            "🔴 VERY LOW CONSIDERATION — "
            "The model identifies a very high-risk profile."
        )


    # ========================================================
    # APPROVAL / REJECTION
    # ========================================================

    st.markdown(
        "### 🏦 LOAN DECISION"
    )


    if probability >= approval_threshold:

        st.success(
            f"""
            ### ✅ LOAN APPROVED

            **Predicted probability:** {probability * 100:.1f}%

            **Required threshold:** {approval_threshold * 100:.0f}%

            The predicted probability is above the configured
            approval threshold.
            """
        )

        st.balloons()

    else:

        st.error(
            f"""
            ### ❌ LOAN NOT APPROVED

            **Predicted probability:** {probability * 100:.1f}%

            **Required threshold:** {approval_threshold * 100:.0f}%

            The predicted probability is below the configured
            approval threshold.
            """
        )


    # ========================================================
    # SCORE INTERPRETATION
    # ========================================================

    st.markdown(
        "### 🧠 AI ASSESSMENT SUMMARY"
    )


    summary_col1, summary_col2 = st.columns(2)


    with summary_col1:

        st.info(
            f"""
            **Creditworthiness Score**

            `{credit_score}/100`

            This score represents the model's predicted
            probability of the applicant belonging to the
            good-credit class.
            """
        )


    with summary_col2:

        st.info(
            f"""
            **Consideration Level**

            `{consideration}`

            This is an interpretation layer built on top
            of the ML probability. It is not an official
            banking or CIBIL score.
            """
        )


    # ========================================================
    # TECHNICAL DETAILS
    # ========================================================

    with st.expander(
        "🔧 VIEW TECHNICAL MODEL DETAILS"
    ):

        technical_col1, technical_col2 = st.columns(2)


        with technical_col1:

            st.write(
                "**Machine Learning Model:** "
                "Logistic Regression"
            )

            st.write(
                "**Dataset:** "
                "German Credit Dataset"
            )

            st.write(
                "**Training Samples:** "
                f"{model_info['training_samples']}"
            )

            st.write(
                "**Testing Samples:** "
                f"{model_info['testing_samples']}"
            )


        with technical_col2:

            st.write(
                "**Accuracy:** "
                f"{model_info['accuracy'] * 100:.2f}%"
            )

            st.write(
                "**ROC-AUC:** "
                f"{model_info['roc_auc']:.4f}"
            )

            st.write(
                "**Model Features:** "
                f"{model_info['number_of_features']}"
            )

            st.write(
                "**Approval Threshold:** "
                f"{approval_threshold}"
            )


# ============================================================
# FOOTER
# ============================================================

st.markdown(
    """
    <div class="footer">

        ⚡ SMART LOAN AI • MACHINE LEARNING CREDIT INTELLIGENCE ⚡

        <br><br>

        Logistic Regression • German Credit Dataset

    </div>
    """,
    unsafe_allow_html=True
)

Overwriting /content/app.py


In [27]:
# ============================================================
# CELL 18: START STREAMLIT
# ============================================================

!pkill -f "streamlit run /content/app.py" || true

!streamlit run /content/app.py \
    --server.port 8501 \
    --server.address 0.0.0.0 \
    > /content/streamlit.log 2>&1 &

import time

time.sleep(5)

print("=" * 70)
print("STREAMLIT SERVER STARTED")
print("=" * 70)

!cat /content/streamlit.log

^C
STREAMLIT SERVER STARTED


2026-07-29 17:34:00.043 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.26.54.168:8501



In [28]:
# ============================================================
# CELL 19: START CLOUDFLARE TUNNEL
# ============================================================

!pkill -f cloudflared || true

!cloudflared tunnel \
    --url http://localhost:8501 \
    > /content/cloudflare.log 2>&1 &

import time

time.sleep(8)

print("=" * 70)
print("CLOUDFLARE TUNNEL")
print("=" * 70)

!cat /content/cloudflare.log

^C
CLOUDFLARE TUNNEL
2026-07-29T17:34:06Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-29T17:34:06Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-29T17:34:10Z INF +--------------------------------------------------------------------------------------------+
2026-07-29T17:34:10Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-29T17:34:10Z INF |  https://asset-processing-vinyl-f